# EurecomGPT — Phase 3: MapReduce & Spark for the RAG corpus

Run in **Google Colab** (CPU runtime is fine). Phase 3 builds a TF-IDF index two ways. The **MapReduce** half already ran — in Cloud Shell, as Cloud Run function tasks orchestrated by Cloud Workflows (`run_mr.py`, TASKS.md Tasks 3–6). This notebook picks up its result, then does the **Spark** half locally and lands the TF-IDF table in Cloud Storage and BigQuery.

**Before you start:** `submission/phase3_mapreduce.json` must be committed and pushed — section 1 reads it from your clone. Run the cells **in order**: the report cell at the end collects variables from every section.

## 0. Setup — clone your repo and install dependencies

Colab starts empty, so pull in your repository — that is where your `spark_tfidf.py` and the MapReduce result live. `pyspark` is the only heavy install; the Google client libraries are for sections 4–5.

In [ ]:
# Edit the URL to YOUR repo — the notebook imports your code from it.
!git clone https://github.com/<you>/<your-repo>.git repo 2>/dev/null || true
%cd repo
!pip -q install pyspark pyarrow google-cloud-storage google-cloud-bigquery
# Phase-3 modules are in a subfolder, so add it to the import path.
import sys; sys.path.insert(0, 'phase-3-mapreduce-spark')

### Imports and the corpus

- `corpus.load_documents()` — the same deterministic 60-document TinyShakespeare split that `run_mr.py` used, so the two halves of the phase index the *same* documents.
- `mapreduce` — your primitives, used here only as the local reference.
- `spark_tfidf` — your two RDD stages; `report` — the writer for section 6.

**What to look for:** `documents: 60`.

In [ ]:
import json, time
import mapreduce, corpus, spark_tfidf, report

docs = corpus.load_documents()   # deterministic: TinyShakespeare split into documents
print('documents:', len(docs))

## 1. Your cloud MapReduce result (Lecture 5)

`run_mr.py` wrote `submission/phase3_mapreduce.json` when your Workflows job finished: the execution name, one record per map and reduce task, the elapsed time, and the top terms. This cell loads it and runs the **local reference** — the same `map_wc`, `shuffle`, `reduce_wc` in one process — so you can put the two side by side.

**What to look for:** the top terms must be *identical* (a `MapReduce result mismatch` error means your deployment and your local code disagree). Then compare the two elapsed times: the cloud job is **orders of magnitude slower** on this corpus. That is not a bug — it is per-task overhead (function invocations, Cloud Storage round trips, workflow scheduling) on a job far too small to amortise it. Estimate for the writeup how big the corpus would have to be before the cloud version wins.

In [ ]:
MR_PATH = 'submission/phase3_mapreduce.json'
try:
    mr_cloud = json.load(open(MR_PATH))
except FileNotFoundError:
    raise SystemExit(f'{MR_PATH} not found in your clone — run run_mr.py in Cloud Shell '
                     '(TASKS.md Task 6), commit the file, push, and re-run section 0.')

# Local reference: same primitives, one process, no network.
t0 = time.perf_counter()
counts = mapreduce.word_count(docs)
local_s = time.perf_counter() - t0
top = report.top_terms(counts, n=10)

print(f"cloud job : {mr_cloud['num_splits']} map tasks, {mr_cloud['num_reducers']} reduce tasks, "
      f"{mr_cloud['cloud_elapsed_s']} s, state {mr_cloud['state']}")
print(f'local run : {local_s*1000:.0f} ms')
print('vocab size:', len(counts), '| top terms:', top)
assert [list(x) for x in mr_cloud['top_terms']] == top, 'MapReduce result mismatch: cloud vs local'

## 2. PySpark TF-IDF (Lecture 6)

The same corpus, now as a chain of RDD **transformations** (`flatMap`, `reduceByKey`, `join`, `map`) ending in one **action** (`collect`). Nothing computes until that action runs — the earlier lines only build the lineage graph. `local[*]` means one JVM using all Colab cores; the first cell is slow because it starts that JVM.

**What to look for:** a few thousand `tf-idf rows` and sample rows that look like `{'term': ..., 'doc_id': ..., 'tf': 2, 'df': 7, 'idf': 2.15, 'tfidf': 4.3}`. Time this too and compare with section 1 — Spark's start-up cost is the same lesson again.

In [ ]:
from pyspark.sql import SparkSession

# One local Spark: driver + executors inside this Colab VM. setLogLevel hides the noise.
spark = SparkSession.builder.master('local[*]').appName('tfidf').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

t0 = time.perf_counter()
docs_rdd = spark.sparkContext.parallelize(docs)                  # transformation-free: just distributes the list
rows = spark_tfidf.build_tfidf_rows(docs_rdd, len(docs))        # your term_freq + doc_freq, then join + collect
spark_s = time.perf_counter() - t0
print(f'tf-idf rows: {len(rows)} in {spark_s:.1f} s'); print(rows[:3])

Write the table as **Parquet** — a columnar format BigQuery loads natively (section 5).

In [ ]:
spark_tfidf.save_parquet(rows, 'tfidf.parquet')   # pyarrow, columns term/doc_id/tf/df/idf/tfidf
print('wrote tfidf.parquet')

## 3. Transformations, actions, lineage, and the two runtimes (writeup)

In `submission/phase3_comparison.md`, answer:

1. Which pipeline steps are **transformations** and which are **actions**? Where does Spark's **lineage** let it recover a lost partition without recomputing everything?
2. In your cloud MapReduce job, what played the role of Hadoop's *JobTracker*, its *workers*, and *HDFS*? Why must a map task be **idempotent** for the workflow's retry policy to be safe? (If you ran with `--chaos`, describe what you saw in the execution log.)
3. Compare the three runs — local reference, cloud MapReduce, local Spark — on wall time. Explain the differences, and estimate the corpus size at which the cloud job would break even.

## 4. Upload the Parquet to Cloud Storage (public)

Colab is not logged into `gcloud`, so authenticate with your Google account first, then use the Python client. The bucket is the one `run_mr.py` already created and made public — the same `<project>-eurecomgpt` bucket Phase 5 will read from.

In [ ]:
from google.colab import auth
auth.authenticate_user()   # opens a popup to log into your Google account

**Set `PROJECT`** to your Phase-0 project id. The IAM step is idempotent — re-adding the `allUsers` binding is harmless if `run_mr.py` already did it.

**What to look for:** the printed URL downloads in a browser.

In [ ]:
PROJECT = 'REPLACE-with-your-project-id'   # <-- your Phase-0 GCP project id
BUCKET  = f'{PROJECT}-eurecomgpt'

from google.cloud import storage
client = storage.Client(project=PROJECT)
try:
    bucket = client.get_bucket(BUCKET)
except Exception:
    bucket = client.create_bucket(BUCKET, location='US')
# Public read via bucket IAM (works with uniform bucket-level access).
policy = bucket.get_iam_policy(requested_policy_version=3)
policy.bindings.append({'role': 'roles/storage.objectViewer', 'members': {'allUsers'}})
bucket.set_iam_policy(policy)
bucket.blob('tfidf.parquet').upload_from_filename('tfidf.parquet')
PARQUET_URL = f'https://storage.googleapis.com/{BUCKET}/tfidf.parquet'
print('public URL:', PARQUET_URL)

## 5. Load the TF-IDF table into BigQuery and query it

The lakehouse layer: BigQuery loads the Parquet straight from Cloud Storage (no rows pass through this notebook), then a SQL query does retrieval over it — the same query shape Phase 5's chat app will run for every user prompt.

**What to look for:** ten `[term, doc_id, tfidf]` rows, highest tf-idf first — rare, document-specific words (names, places), not common ones.

In [ ]:
from google.cloud import bigquery
bq = bigquery.Client(project=PROJECT)
DATASET = f'{PROJECT}.eurecomgpt'
bq.create_dataset(bigquery.Dataset(DATASET), exists_ok=True)
TABLE = f'{DATASET}.tfidf'

# Server-side load: BigQuery reads the Parquet from GCS itself.
job = bq.load_table_from_uri(
    f'gs://{BUCKET}/tfidf.parquet', TABLE,
    job_config=bigquery.LoadJobConfig(source_format=bigquery.SourceFormat.PARQUET,
                                      write_disposition='WRITE_TRUNCATE'))
job.result()   # blocks until the load finishes

q = f'SELECT term, doc_id, tfidf FROM `{TABLE}` ORDER BY tfidf DESC LIMIT 10'
top_by_tfidf = [[r.term, r.doc_id, round(r.tfidf, 4)] for r in bq.query(q).result()]
print('top by tf-idf:', top_by_tfidf)

## 6. Write the submission report

Collects everything into `submission/phase3_report.json`: the cloud MapReduce record from section 1 (plus your local timing), the Spark row count and Parquet URL from sections 2 and 4, and the BigQuery table and query result from section 5. A `NameError` here names the section you skipped.

In [ ]:
report.write_report(
    corpus_info={'num_docs': len(docs)},
    mapreduce={**mr_cloud, 'local_elapsed_s': round(local_s, 3)},
    tfidf={'parquet_gcs_url': PARQUET_URL, 'num_rows': len(rows), 'spark_elapsed_s': round(spark_s, 1)},
    bigquery={'table': TABLE, 'top_by_tfidf': top_by_tfidf},
    comparison={'notes': 'see phase3_comparison.md'},
)

Now **commit** `submission/phase3_report.json` (+ your `phase3_comparison.md`) and push. Keep `wordcount.json` and `tfidf.parquet` public in the bucket until you are graded.